In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
from torch.autograd import Variable
from torchvision.datasets import MNIST
from torchvision import transforms

In [2]:
# Define the autoencoder model with bilinear interpolation
class Autoencoder(nn.Module):
    def __init__(self):
        super(Autoencoder, self).__init__()

        self.encoder = nn.Sequential(
            nn.Linear(784, 256),
            nn.ReLU(),
            nn.Linear(256, 128),
            nn.ReLU(),
            nn.Linear(128, 64),
            nn.ReLU()
        )

        self.decoder = nn.Sequential(
            nn.Linear(64, 128),
            nn.ReLU(),
            nn.Linear(128, 256),
            nn.ReLU(),
            nn.Linear(256, 784),
            nn.Sigmoid()
        )

    def forward(self, x):
        encoded = self.encoder(x)
        decoded = self.decoder(encoded)
        return decoded

# Create a custom dataset class
class MyDataset(data.Dataset):
    def __init__(self, data, labels):
        self.data = data
        self.labels = labels

    def __getitem__(self, index):
        x = self.data[index]
        y = self.labels[index]
        return x, y

    def __len__(self):
        return len(self.data)

In [3]:
# Define the training function
def train_autoencoder(model, dataloader, num_epochs=10, learning_rate=0.001):
    criterion = nn.MSELoss()
    optimizer = optim.Adam(model.parameters(), lr=learning_rate)

    for epoch in range(num_epochs):
        total_loss = 0.0

        for data, _ in dataloader:
            data = Variable(data.view(data.size(0), -1))

            # Zero the gradients
            optimizer.zero_grad()

            # Forward pass
            output = model(data)

            # Compute the loss
            loss = criterion(output, data)

            # Backward pass and optimize
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print('Epoch [{}/{}], Loss: {:.4f}'.format(epoch + 1, num_epochs, total_loss / len(dataloader)))

In [4]:
# Load the MNIST dataset
transform = transforms.Compose([transforms.ToTensor(), transforms.Normalize((0.5,), (0.5,))])
train_dataset = MNIST(root='./data', train=True, download=True, transform=transform)
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=64, shuffle=True)

In [5]:
# Create an instance of the autoencoder model
model = Autoencoder()

In [6]:
# Train the autoencoder
train_autoencoder(model, train_loader)

Epoch [1/10], Loss: 0.9329
Epoch [2/10], Loss: 0.9254
Epoch [3/10], Loss: 0.9254
Epoch [4/10], Loss: 0.9254
Epoch [5/10], Loss: 0.9254
Epoch [6/10], Loss: 0.9254
Epoch [7/10], Loss: 0.9254
Epoch [8/10], Loss: 0.9254
Epoch [9/10], Loss: 0.9254
Epoch [10/10], Loss: 0.9254


In [7]:
# Create a tensor with random noise for the input
input_data = torch.randn(1, 784)

In [14]:
encoded

tensor([[0.3112, 0.8754, 0.3320, 0.4274, 0.0000, 0.3006, 0.2061, 0.0000, 0.0000,
         0.0000, 0.6874, 0.0000, 0.0000, 0.0177, 0.0000, 0.0000, 0.5267, 0.0000,
         0.1661, 0.0000, 0.0000, 0.4713, 0.6370, 0.5481, 1.3009, 0.0000, 0.0363,
         0.8521, 0.4347, 0.0000, 0.5294, 0.0000, 0.0000, 0.0000, 0.0000, 0.5429,
         0.0000, 0.6768, 0.5391, 0.6937, 0.7998, 0.2980, 0.0000, 0.0000, 0.8282,
         0.0000, 0.0000, 0.6763, 0.4370, 0.2776, 0.0000, 0.0000, 0.0000, 0.2645,
         0.0000, 0.9710, 0.0000, 0.2814, 0.7754, 0.0000, 0.0000, 0.0000, 1.0154,
         0.5415]], grad_fn=<ReluBackward0>)

In [15]:
# Get the encoded representation of the input
encoded = model.encoder(input_data)

In [16]:
# Modify the encoded tensor to match the desired label (5)
encoded[0][-1] = 5

In [17]:
# Reconstruct the data from the modified encoded tensor
reconstructed = model.decoder(encoded)

# Reshape the reconstructed data into an image
reconstructed_image = reconstructed.view(28, 28).detach().numpy()

In [18]:
reconstructed_image

array([[0.06573367, 0.06739265, 0.07513101, 0.07210507, 0.03626952,
        0.09096169, 0.03194709, 0.07593896, 0.07237   , 0.06133213,
        0.07844677, 0.06831289, 0.05603244, 0.07434225, 0.06562532,
        0.06045027, 0.03253065, 0.04709602, 0.06112784, 0.05276367,
        0.08861541, 0.03403284, 0.04623819, 0.09193496, 0.04746887,
        0.03626123, 0.03690591, 0.0374945 ],
       [0.06987698, 0.0524537 , 0.04133112, 0.03559817, 0.08271306,
        0.05343474, 0.03933028, 0.08099013, 0.06988088, 0.04214079,
        0.04822782, 0.0625051 , 0.06216268, 0.04918386, 0.07232194,
        0.07122337, 0.05285573, 0.05835588, 0.05319978, 0.03001594,
        0.04917094, 0.06457153, 0.05333199, 0.05655885, 0.06373524,
        0.0640271 , 0.08095161, 0.07003672],
       [0.05996648, 0.0746946 , 0.05209329, 0.0306969 , 0.07034374,
        0.07231849, 0.06342757, 0.06760859, 0.07285486, 0.05453728,
        0.08776156, 0.02797015, 0.06635171, 0.09132317, 0.05190052,
        0.08206748, 0.0503

In [ ]:
# Display the reconstructed image
import matplotlib.pyplot as plt
plt.imshow(reconstructed_image, cmap='gray')
plt.axis('off')
plt.show()

In [ ]:
import matplotlib.pyplot as plt

# Plot the reconstructions
fig, axes = plt.subplots(nrows=2, ncols=5, figsize=(10, 4))

for i, ax in enumerate(axes.flatten()):
    ax.imshow(reconstructed_image, cmap='gray')
    ax.axis('off')

plt.tight_layout()
plt.show()